In [6]:
import pandas as pd

import tree_sitter_cypher
from tree_sitter import Language, Parser

In [7]:
_CYPHER_LANGUAGE: Language = Language(tree_sitter_cypher.language())
_default_parser: Parser = Parser(_CYPHER_LANGUAGE)

In [8]:
df = pd.read_csv('../data/processed/text-to-cypher/CypherBench/train.csv')
df

,qid,graph,gold_cypher,nl_question,answer_json,from_template,schema,sample
0,f592b90d-a4b9-43b9-ae5b-b7e9d5792c72,biology,MATCH (n:Taxon)-[r0:inhabits]->(m0:Habitat {na...,What are the names of taxa that live in floodp...,"[[""Equisetum hyemale"", null]]","{'match_category': 'basic_(n)-(m0*)', 'match_c...",<schema:>\n <node:> Taxon\n <prop:> ...,<cmd:> What are the names of taxa that live in...
1,e74fa5ca-1593-47b0-867b-39c6833778c9,soccer,MATCH (n:Player)-[r0:playsPosition]->(m0:Posit...,What are the names and dates of death of playe...,"[[""Ricardo Esgaio"", null], [""Vasco Matos"", nul...","{'match_category': 'basic_(n)-(m0*),(n)-(m1*)'...",<schema:>\n <node:> Player\n <prop:>...,<cmd:> What are the names and dates of death o...
2,d88fb5a8-94b4-4492-9fdf-c8fc26a8712a,soccer,MATCH (n:Player)-[r0:playsPosition]->(m0:Posit...,Who is the youngest player in the attacking mi...,"[[""Kutlwelo Mpolokang""]]","{'match_category': 'basic_(n)-(m0*)', 'match_c...",<schema:>\n <node:> Player\n <prop:>...,<cmd:> Who is the youngest player in the attac...
3,4fde185b-f894-44eb-8bd0-a1a2a1057f73,terrorist_attack,MATCH (n:TerroristAttack)-[r0:targets]->(m0:Ta...,What are the names and injury counts of terror...,"[[""September 11 attacks"", 25000]]","{'match_category': 'basic_(n)-(m0)-(m1*)', 'ma...",<schema:>\n <node:> TerroristAttack\n ...,<cmd:> What are the names and injury counts of...
4,e7b12747-9b40-48e1-8294-336d9e73cc9f,terrorist_attack,MATCH (n:TerroristAttack)-[r0:targets]->(m0:Ta...,What are the names and dates of terrorist atta...,"[[""26/11 Mumbai attacks"", ""2008-11-26""]]","{'match_category': 'basic_(n)-(m0*),(n)-(m1*)'...",<schema:>\n <node:> TerroristAttack\n ...,<cmd:> What are the names and dates of terrori...
...,...,...,...,...,...,...,...,...
8529,6ea8c3e6-1780-4e0d-aef5-9539e7fc2058,terrorist_attack,CALL { MATCH (n:TerroristAttack)-[r0:employs]-...,What are the names of terrorist attacks that u...,"[[""November 2020 Afghanistan attacks""], [""1990...","{'match_category': 'special_union', 'match_cyp...",<schema:>\n <node:> TerroristAttack\n ...,<cmd:> What are the names of terrorist attacks...
8530,81bdac31-29ba-444b-8762-767fbe77b24a,soccer,MATCH (n:Club)<-[r0:playsFor]-(m0:Player {name...,What are the names of the clubs Serge Gakpé ha...,"[[""Tours FC."", ""Ren\u00e9 Lobello""], [""Genoa C...","{'match_category': 'basic_(n)-(m0*)', 'match_c...",<schema:>\n <node:> Player\n <prop:>...,<cmd:> What are the names of the clubs Serge G...
8531,405e343a-b5e9-4b81-a888-972b4d4f0c67,soccer,MATCH (n:Player)-[r0:playsFor]->(m0:Club)<-[r1...,What are the different genders of players who ...,"[[""male""], [null]]","{'match_category': 'basic_(n)-(m0)-(m1*)', 'ma...",<schema:>\n <node:> Player\n <prop:>...,<cmd:> What are the different genders of playe...
8532,ed0ad361-f804-4bb4-aa2d-813a23a2f769,biology,MATCH (n:ConservationStatus)<-[r0:hasConservat...,What are the conservation statuses of taxa tha...,"[[""Least Concern""], [""Data Deficient""]]","{'match_category': 'basic_(n)-(m0)-(m1*)', 'ma...",<schema:>\n <node:> Taxon\n <prop:> ...,<cmd:> What are the conservation statuses of t...


In [9]:
# count how many queries can be parsed by tree-sitter-cypher
def is_cypher_valid(query: str) -> bool:
    tree = _default_parser.parse(bytes(query, "utf8"))
    root_node = tree.root_node
    return root_node.has_error == False and root_node.child_count > 0

df['is_valid'] = df['gold_cypher'].apply(is_cypher_valid)
valid_count = df['is_valid'].sum()
total_count = len(df)
print(f"Valid Cypher queries: {valid_count}/{total_count} ({valid_count / total_count * 100:.2f}%)")

Valid Cypher queries: 7587/8534 (88.90%)


In [10]:
# print non valid queries
df[df['is_valid'] == False]

,qid,graph,gold_cypher,nl_question,answer_json,from_template,schema,sample,is_valid
18,a07f74eb-d0ac-4718-b400-059dceb765a7,terrorist_attack,CALL { MATCH (n:TerroristAttack)-[r0:employs]-...,What are the names of terrorist attacks that u...,"[[""2023 Hamas-led attack on Israel""], [""2017 S...","{'match_category': 'special_union', 'match_cyp...",<schema:>\n <node:> TerroristAttack\n ...,<cmd:> What are the names of terrorist attacks...,False
33,66cac439-4cda-4da9-aa1e-d7e2d68e3315,biology,CALL { MATCH (n:ConservationStatus)<-[r0:hasCo...,What are the conservation statuses of either t...,"[[""Least Concern""]]","{'match_category': 'special_union', 'match_cyp...",<schema:>\n <node:> Taxon\n <prop:> ...,<cmd:> What are the conservation statuses of e...,False
37,51d34804-d4fc-4cdf-9259-2416d9474ac8,biology,CALL { MATCH (n:Taxon)-[r0:feedsOn]->(m0:Taxon...,How many taxa feed on either the Ovibos moscha...,[[1]],"{'match_category': 'special_union', 'match_cyp...",<schema:>\n <node:> Taxon\n <prop:> ...,<cmd:> How many taxa feed on either the Ovibos...,False
39,d625a688-bfe5-427b-8700-2b76f5f8d41e,terrorist_attack,CALL { MATCH (n:TerroristAttack)-[r0:employs]-...,How many terrorist attacks either used a grena...,[[32]],"{'match_category': 'special_union', 'match_cyp...",<schema:>\n <node:> TerroristAttack\n ...,<cmd:> How many terrorist attacks either used ...,False
54,5f3d5705-2c8f-4109-a1be-12ba2b07db63,art,CALL { MATCH (n:Genre)<-[r0:hasGenre]-(m0:Scul...,What are the names of genres associated with e...,"[[""public art""], [""mythological sculpture""]]","{'match_category': 'special_union', 'match_cyp...",<schema:>\n <node:> Person\n <prop:>...,<cmd:> What are the names of genres associated...,False
...,...,...,...,...,...,...,...,...,...
8515,da4056d8-57a6-4aca-86d0-aee4f2a7202d,terrorist_attack,CALL { MATCH (n:TerroristAttack)-[r0:perpetrat...,How many terrorist attacks were carried out by...,[[1]],"{'match_category': 'special_union', 'match_cyp...",<schema:>\n <node:> TerroristAttack\n ...,<cmd:> How many terrorist attacks were carried...,False
8519,5b657c43-b67d-4d79-a561-ce068a344c30,art,CALL { MATCH (n:Sculpture)-[r0:associatedWith]...,How many sculptures are either linked to the I...,[[27]],"{'match_category': 'special_union', 'match_cyp...",<schema:>\n <node:> Person\n <prop:>...,<cmd:> How many sculptures are either linked t...,False
8521,ebcd2158-0f7e-4df1-8bd4-4d5eadcdefad,terrorist_attack,CALL { MATCH (n:TerroristAttack)-[r0:employs]-...,How many terrorist attacks involved either a s...,[[13]],"{'match_category': 'special_union', 'match_cyp...",<schema:>\n <node:> TerroristAttack\n ...,<cmd:> How many terrorist attacks involved eit...,False
8528,3cf52daf-feeb-4a14-a3d0-672c8d216287,biology,CALL { MATCH (n:Taxon)<-[r0:feedsOn]-(m0:Taxon...,What are the names of taxa that are either pre...,"[[""roe deer""], [""Nyctereutes procyonoides""], [...","{'match_category': 'special_union', 'match_cyp...",<schema:>\n <node:> Taxon\n <prop:> ...,<cmd:> What are the names of taxa that are eit...,False


In [ ]:
# get unique graph label values
df['graph'].unique()

In [ ]:
# print all unique special tokens in sample
special_tokens = set()
for sample in df['sample']:
    tokens = [token for token in sample.split() if ">" in token]
    special_tokens.update(tokens)
print(special_tokens)